In [96]:
%pip install pandas
%pip install python-calamine

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [97]:
import pandas as pd 
import numpy as np
import re

In [116]:
data = pd.read_excel(r'C:\Users\USER\Desktop\สต็อกสินค้า1.xlsx' ,engine='calamine',header=11,usecols="A:F",dtype={'Unnamed: 3': str})
data.rename(columns={'Unnamed: 0':'DATE','Unnamed: 1':'Bill','เพิ่ม ':'details','Unnamed: 3':'value','ลด ':'export','คงเหลือ ':'balance'} ,inplace=True)

In [117]:
data['product_id'] = data.loc[data['DATE'] =='รหัสสินค้า', 'export']
data['product_id'] = data['product_id'].ffill()

In [118]:
data['unit'] = data.loc[data['DATE'].astype(str).str.strip() == 'คลัง', 'balance']
data['unit'] = data['unit'].ffill()
data['unit'] = data['unit'].str.extract(r'(\d+)').fillna(0).astype(int)

In [119]:
# แปลงตรงๆ โดยบอกสไตล์ปฏิทินสากลไปก่อน
data['DATE'] = pd.to_datetime(data['DATE'], format='%d/%m/%Y', errors='coerce')

# ลบปีออก 543 ปี (ใช้ DateOffset)
data['DATE'] = data['DATE'] - pd.DateOffset(years=543)
data['DATE'] = data['DATE'].dt.date

In [120]:
data.dropna(subset=['DATE'], inplace=True)

In [121]:
# 1. บังคับชุบชีวิตข้อมูลในคอลัมน์ให้กลายเป็นข้อความ (String) ชัวร์ๆ 100% ก่อน
data['value_str'] = data['value'].astype(str)

# 2. ก่อนรัน Regex เติมจุดทศนิยม .0 เข้าไปเฉพาะตัวที่เป็นจำนวนเต็ม (เช่น '5' -> '5.0')
# เพื่อป้องกันไม่ให้ Regex ตัวหน้าและตัวหลังพัง
data['value_str'] = data['value_str'].apply(lambda x: x if '.' in x else x + '.0')

# 3. เอาตัวเลขหน้าจุด (ใช้ตัวแปร value_str ที่เป็นข้อความแล้ว)
data['front_value'] = data['value_str'].str.extract(r"^([0-9]+)\.").fillna(0).astype(int)

# 4. เอาตัวเลขหลังจุด (ข้าม 0 หน้า เจอเลข 1-9 ปุ๊บ กวาดขวาหมดเท่าที่มีจริง)
data['back_value'] = data['value_str'].str.extract(r"\.0*([1-9].*)").fillna(0).astype(int)

In [122]:
data['import'] = data['back_value'] + (data['unit'] * data['front_value']) 

In [123]:
data

,DATE,Bill,details,value,export,balance,product_id,unit,value_str,front_value,back_value,import
2,2026-03-01,.,ยอดยกมา,20,0,20,กะซ้ง น้ำดื่ม 850 มล.,1,20.0,20,0,20
3,2026-03-01,690301/100310001,ใบขายสด-Pos ขายให้.เงินสด,0,7,13,กะซ้ง น้ำดื่ม 850 มล.,1,0.0,0,0,0
4,2026-03-01,690301/100320001,ใบขายสด-Pos ขายให้.เงินสด,0,1,12,กะซ้ง น้ำดื่ม 850 มล.,1,0.0,0,0,0
5,2026-03-02,690302/100310001,ใบขายสด-Pos ขายให้.เงินสด,0,2,10,กะซ้ง น้ำดื่ม 850 มล.,1,0.0,0,0,0
6,2026-03-03,IBK3256903/004,ใบรับสินค้าจากการซื้อK3 ซื้อจาก.หจก. กะซ้งค้าส่ง,120,0,130,กะซ้ง น้ำดื่ม 850 มล.,1,120.0,120,0,120
...,...,...,...,...,...,...,...,...,...,...,...,...
7565,2026-05-02,DM03256905/005,โอนสินค้าสนง.ใหญ่ ไป - Kmart3 รับย้ายจาก.00,1,0,1.4,SMEซอง ลดปัญหาฝ้าแดด /ไวท์เบบี้เฟซ เซรั่ม,6,1.0,1,0,6
7566,2026-07-02,690702/100310001,ใบขายสด-Pos ขายให้.เงินสด,0,0.2,1.2,SMEซอง ลดปัญหาฝ้าแดด /ไวท์เบบี้เฟซ เซรั่ม,6,0.0,0,0,0
7567,2026-07-04,690704/100310001,ใบขายสด-Pos ขายให้.เงินสด,0,0.1,1.1,SMEซอง ลดปัญหาฝ้าแดด /ไวท์เบบี้เฟซ เซรั่ม,6,0.0,0,0,0
7568,2026-07-08,690708/100310001,ใบขายสด-Pos ขายให้.เงินสด,0,0.1,1,SMEซอง ลดปัญหาฝ้าแดด /ไวท์เบบี้เฟซ เซรั่ม,6,0.0,0,0,0


 Robust Z-score

In [ ]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement scipy (from versions: none)

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for scipy


In [ ]:
%pip install python-calamine
%pip install pandas
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [108]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import openpyxl

# ==========================================
# 1. โหลดข้อมูล (ใส่ engine='calamine' เพื่อเลี่ยง XML เสียหายจากรอบแรก)
# ==========================================

df = data[['DATE', 'Bill', 'details', 'product_id', 'import']].copy()

# ล้างช่องว่างที่อาจมองไม่เห็นในชื่อคอลัมน์ทิ้งให้หมดเพื่อความปลอดภัย
df.columns = df.columns.str.strip()

# [เสริมเกราะ 1] แปลง ID ให้เป็น string ทั้งหมด ป้องกันกรณี Excel แปลงบางตัวเป็นตัวเลขแล้วกลุ่มเพี้ยน
df['product_id'] = df['product_id'].astype(str).str.strip()

# เอาเฉพาะยอดนำเข้าที่มากกว่า 0 เท่านั้น (ตัด Noise/บิลยกเลิก ออก)
df = df[df['import'] > 0]

# ==========================================
# 2. คำนวณหา Outlier 
# ==========================================
# หา IQR และคูณสเกล 1.4826 สำหรับ แผน A
q1 = df.groupby('product_id')['import'].transform(lambda x: x.quantile(0.25))
q3 = df.groupby('product_id')['import'].transform(lambda x: x.quantile(0.75))
iqr_scaled = (q3 - q1) * 1.4826

# หา Median และจำนวนบิล
group_median = df.groupby("product_id")["import"].transform("median")
group_count = df.groupby('product_id')['import'].transform('count')
group_sd = df.groupby("product_id")["import"].transform("std")
# ✨ [แก้ไขจุดที่ 1] หา MAD ดิบจาก SciPy แล้วค่อยคูณสเกล 1.4826 ข้างนอก 
mad_raw = df.groupby("product_id")["import"].transform(stats.median_abs_deviation)
group_mad_scaled = mad_raw * 1.4826

# เงื่อนไขที่ 2 ไม่ให้สูงเกิน 30% ของค่ากลาง
max_allowed_deviation = np.maximum(group_median * 0.3, 1.0)
group_mad_scaled = np.minimum(group_mad_scaled, max_allowed_deviation)
group_mad_scaled = np.maximum(group_mad_scaled, 1.0) # กันตัวหารเป็น 0

# เงื่อนไขการแบ่งกลุ่มสินค้า
conditions = [
    (iqr_scaled > 0) & (group_count >= 10),   # แผน A
    (iqr_scaled == 0) | (group_count < 10)    # แผน B
]

# คำนวณคะแนนดิบ Z-score
df["Adaptive_ZScore"] = np.select(conditions, [
    ((df["import"] - group_median) / iqr_scaled),       # แผน A 
    ((df["import"] - group_median) / group_mad_scaled)  # แผน B 
], default=0)

# min max mean 
df['min'] = df.groupby('product_id')['import'].transform('min')
df['max'] = df.groupby('product_id')['import'].transform('max')
df['mean'] = df.groupby('product_id')['import'].transform('mean')
df["z_score"] = (df["import"] - df['mean']) / group_sd
df["median"] = group_median
df["IQR"] = (q3 - q1)


# ==============================================================================
# 📊 สเกล Z-Score / SD แปลงเป็นเปอร์เซ็นต์ข้อมูลที่ครอบคลุม (Normal Distribution)
# ==============================================================================
#  ± 1 SD  = ครอบคลุมข้อมูล  68.27%  (โอกาสหลุดเกณฑ์ ~ 31.73%)
#  ± 2 SD  = ครอบคลุมข้อมูล  95.45%  (โอกาสหลุดเกณฑ์ ~  4.55%)
#  ± 3 SD  = ครอบคลุมข้อมูล  99.73%  (โอกาสหลุดเกณฑ์ ~  0.27% -> Outlier)
# ==============================================================================
df["is_outlier"] = df['Adaptive_ZScore'].abs() > 3

# ==========================================
# 3. เจาะลึกระดับบิล (Drill-Down)
# ==========================================
# ดึงรายชื่อรหัสสินค้าทั้งหมดที่มีแถวใดแถวหนึ่งติดสถานะ Outlier
final_report = df[df["is_outlier"] == True].copy()

# ✨ [แก้ไขจุดที่ 2] จัดเรียงข้อมูลพร้อมใส่วงเล็บปิดให้สมบูรณ์
final_report = final_report.sort_values(
    by=["product_id", "Adaptive_ZScore"], ascending=[True, False]
)

In [ ]:
#final_report.to_excel(r'C:\Users\KS\Desktop\product_out1.xlsx',index=False,engine='openpyxl')

In [109]:
final_report[['DATE','Bill','product_id','import','min','max','mean','median','IQR','z_score','Adaptive_ZScore']].reset_index(drop=True
                                                                                    )

,DATE,Bill,product_id,import,min,max,mean,median,IQR,z_score,Adaptive_ZScore
0,2026-03-14,IBK3256903/039,กะซ้ง น้ำดื่ม 850 มล.,600,1,600,113.161765,120.0,60.00,5.169963,5.395926
1,2026-03-01,.,กาแฟซุปเปอร์มิก แดง (หีบ30ซอง*30),247,1,247,56.054054,60.0,30.00,4.448507,4.204326
2,2026-06-05,IBK3256906/012,กาแฟซุปเปอร์มิก แดง (หีบ30ซอง*30),240,1,247,56.054054,60.0,30.00,4.285427,4.046945
3,2026-07-12,IBK3256907/031,กาแฟซุปเปอร์มิก แดง (หีบ30ซอง*30),210,1,247,56.054054,60.0,30.00,3.586511,3.372454
4,2026-03-27,IBK3256903/072,กาแฟโกลเด้น หีบ(30ซอง*20),1200,30,1200,83.181818,30.0,30.00,6.065610,26.305140
5,2026-03-14,IBK3256903/039,กาแฟโกลเด้น หีบ(30ซอง*20),450,30,1200,83.181818,30.0,30.00,1.992246,9.442871
6,2026-05-06,IBK3256905/031,คินเดอร์ ทรอนกี้ 18ก.,24,8,24,14.666667,12.0,8.00,1.120897,3.333333
7,2026-03-01,.,ถ่านพานานีโอ ดำAAA กล่อง(2ก้อน*30แพค),32,1,32,5.660377,4.0,6.00,5.098403,3.147624
8,2026-04-13,IBK3256904/044,น้ำ เพียวไลฟ์330,2976,12,2976,373.403509,168.0,168.00,4.026967,11.273631
9,2026-05-13,IBK3256905/039,น้ำ เพียวไลฟ์330,2976,12,2976,373.403509,168.0,168.00,4.026967,11.273631


In [110]:
final_report[['DATE','Bill','product_id','import','min','max','mean','median','IQR','z_score','Adaptive_ZScore']].reset_index(drop=True).to_excel(
    r'C:\Users\USER\Desktop\product_out1.xlsx',index=False,engine='openpyxl')
                                                                                    